In [1]:
from bayes_opt import BayesianOptimization
from bayes_opt import acquisition
import os
import pandas as pd

In [2]:
processed_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed")
input_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "InputData.csv")
output_data_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "OutputData.csv")

df_input = pd.read_csv(input_data_path)
df_output = pd.read_csv(output_data_path)

In [3]:
acq_removal = acquisition.UpperConfidenceBound(kappa=2.5)
acq_recovery = acquisition.UpperConfidenceBound(kappa=2.5)

In [4]:
pbounds = {
    'Flow_C': (min(df_input['Flow_C']), max(df_input['Flow_C'])),
    'Flow_PP': (min(df_input['Flow_PP']), max(df_input['Flow_PP'])),
    'Flow_P': (min(df_input['Flow_P']), max(df_input['Flow_P'])),
    'Current_P': (min(df_input['Current_P']), max(df_input['Current_P']))
}


In [5]:
optimizer_removal = BayesianOptimization(
    f=None,
    acquisition_function=acq_removal,
    pbounds=pbounds,
    verbose=2,
    random_state=1,
    allow_duplicate_points=True
)

optimizer_recovery = BayesianOptimization(
    f=None,
    acquisition_function=acq_recovery,
    pbounds=pbounds,
    verbose=2,
    random_state=1,
    allow_duplicate_points=True
)

In [6]:
for i in range(len(df_input)):
    optimizer_removal.register(
        params={
            'Flow_C': df_input.iloc[i]['Flow_C'],
            'Flow_PP': df_input.iloc[i]['Flow_PP'],
            'Flow_P': df_input.iloc[i]['Flow_P'],
            'Current_P': df_input.iloc[i]['Current_P']
        },
        target=df_output.iloc[i]['Removal']
    )
    optimizer_recovery.register(
        params={
            'Flow_C': df_input.iloc[i]['Flow_C'],
            'Flow_PP': df_input.iloc[i]['Flow_PP'],
            'Flow_P': df_input.iloc[i]['Flow_P'],
            'Current_P': df_input.iloc[i]['Current_P']
        },
        target=df_output.iloc[i]['Recovery']
    )

| 2         | 0.96      | 0.1923076 | 0.2307692 | 0.2307692 | 1.0       |
| 2         | 0.36      | 0.1923076 | 0.2307692 | 0.2307692 | 1.0       |
| 3         | 0.95      | 0.1153846 | 0.1538461 | 0.1538461 | 1.0       |
| 3         | 0.36      | 0.1153846 | 0.1538461 | 0.1538461 | 1.0       |
| 4         | 0.89      | 0.0769230 | 0.0769230 | 0.0769230 | 1.0       |
| 4         | 0.34      | 0.0769230 | 0.0769230 | 0.0769230 | 1.0       |
| 5         | 0.83      | 0.0384615 | 0.0384615 | 0.0384615 | 1.0       |
| 5         | 0.33      | 0.0384615 | 0.0384615 | 0.0384615 | 1.0       |
| 6         | 0.97      | 0.2692307 | 0.3076923 | 0.3076923 | 1.0       |
| 6         | 0.36      | 0.2692307 | 0.3076923 | 0.3076923 | 1.0       |
Data point [0.19230769 0.23076923 0.23076923 1.        ] is not unique. 1 duplicates registered. Continuing ...
| 7         | 0.98      | 0.1923076 | 0.2307692 | 0.2307692 | 1.0       |
Data point [0.19230769 0.23076923 0.23076923 1.        ] is not unique. 1 

In [7]:
next_to_probe_removal = optimizer_removal.suggest()
next_to_probe_recovery = optimizer_recovery.suggest()

In [8]:
def denormalize(norm_values, bounds):
    return {
        k: norm_values[k] * (bounds[k][1] - bounds[k][0]) + bounds[k][0]
        for k in norm_values
    }

input_bounds = {
    'Flow_C': (0.2, 1.5),
    'Flow_PP': (0.2, 1.5),
    'Flow_P': (0.2, 1.5),
    'Current_P': (35, 55)
}

output_bounds = {
    'Removal': (0, 100),
    'Recovery': (0, 100),
}

print("------------------------Next parameters to probe for removal optimization:")
print(denormalize(next_to_probe_removal, input_bounds))

print("------------------------Next parameters to probe for recovery optimization:")
print(denormalize(next_to_probe_recovery, input_bounds))

------------------------Next parameters to probe for removal optimization:
{'Flow_C': np.float64(0.6417957848616105), 'Flow_PP': np.float64(0.6020900627510483), 'Flow_P': np.float64(0.6012498758167224), 'Current_P': np.float64(50.111111466449714)}
------------------------Next parameters to probe for recovery optimization:
{'Flow_C': np.float64(0.34530065204063864), 'Flow_PP': np.float64(0.40030417220396886), 'Flow_P': np.float64(0.3957856796744117), 'Current_P': np.float64(54.864599266952055)}
